# Step 05: Feature Engineering
## MandiMitra ML Pipeline: Short-Term Mandi Price Forecasting

**Objective:** Build clean, predictive, and leak-free input features from historical price and calendar attributes on high-coverage Maharashtra Rice series (`data/processed/maharashtra_rice_modeling_clean.csv`).

**Scope & Rules:**
- Retained Groups:
  1. `APMC Alibagh + Other + Local`
  2. `APMC Murud + Other + Local`
  3. `APMC Palghar + 1009 Kar + Local`
- Zero data leakage: No future observations used in any input feature.
- No interpolation, no forward-filling, no arrival quantity data.
- Lags and rolling metrics are strictly grouped per `[Market, Variety, Grade]` time series.
- Clear separation between input features and prospective **Target Candidates**.



### Setup and Imports


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)

BASE_DIR = Path("..").resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.feature_engineering import (
    sort_time_series,
    create_price_lag_features,
    create_rolling_price_features,
    create_price_momentum_features,
    create_price_range_features,
    create_calendar_features,
    create_target_candidates,
    audit_feature_leakage
)

INPUT_PATH = BASE_DIR / "data" / "processed" / "maharashtra_rice_modeling_clean.csv"
OUTPUT_PATH = BASE_DIR / "data" / "processed" / "maharashtra_rice_features.csv"

df_clean = pd.read_csv(INPUT_PATH)
print(f"Loaded modeling clean dataset: {INPUT_PATH.name}")
print(f"Initial Shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")



Loaded modeling clean dataset: maharashtra_rice_modeling_clean.csv
Initial Shape: 1,687 rows x 13 columns


---
### 1. Chronological Sorting
Ensure each `Market + Variety + Grade` group is strictly sorted chronologically by `Price Date` before computing any sequential or rolling operators.



In [2]:
group_cols = ["Market", "Variety", "Grade"]
df = sort_time_series(df_clean, group_cols=group_cols, date_col="Price Date")

print("Earliest and latest observations per retained group:")
for keys, grp in df.groupby(group_cols):
    print(f"  - {' | '.join(keys)}: {grp['Price Date'].min().strftime('%Y-%m-%d')} to {grp['Price Date'].max().strftime('%Y-%m-%d')} ({len(grp):,} rows)")



Earliest and latest observations per retained group:
  - APMC Alibagh | Other | Local: 2025-01-26 to 2026-09-03 (575 rows)
  - APMC Murud | Other | Local: 2025-01-26 to 2026-09-03 (575 rows)
  - APMC Palghar | 1009 Kar | Local: 2025-01-26 to 2026-09-03 (537 rows)


---
### 2. Price Lag Features
Using `Modal Price` as the primary price variable, create backward lags: `lag_1`, `lag_2`, `lag_3`, `lag_7`, `lag_14`, `lag_30`.

> **Note on Non-Continuous Cadence:**
> Because missing calendar days (Sundays and holidays) are intentionally NOT filled or interpolated, `price_lag_k` represents the price $k$ **previous observed market trading sessions ago**, rather than exact calendar-day offsets.



In [3]:
df = create_price_lag_features(df, group_cols=group_cols, price_col="Modal Price", lags=[1, 2, 3, 7, 14, 30])

lag_cols = [f"price_lag_{k}" for k in [1, 2, 3, 7, 14, 30]]
print("Sample lagged prices for APMC Alibagh:")
display(df[df['Market'] == 'APMC Alibagh'][['Price Date', 'Modal Price'] + lag_cols].head(8))



Sample lagged prices for APMC Alibagh:


,Price Date,Modal Price,price_lag_1,price_lag_2,price_lag_3,price_lag_7,price_lag_14,price_lag_30
0,2025-01-26,3250.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-27,3250.0,3250.0,NaN,NaN,NaN,NaN,NaN
2,2025-01-28,3250.0,3250.0,3250.0,NaN,NaN,NaN,NaN
3,2025-01-29,3250.0,3250.0,3250.0,3250.0,NaN,NaN,NaN
4,2025-01-30,3250.0,3250.0,3250.0,3250.0,NaN,NaN,NaN
5,2025-01-31,3250.0,3250.0,3250.0,3250.0,NaN,NaN,NaN
6,2025-02-01,3250.0,3250.0,3250.0,3250.0,NaN,NaN,NaN
7,2025-02-02,3250.0,3250.0,3250.0,3250.0,3250.0,NaN,NaN


---
### 3. Rolling Price Features (Moving Averages & Volatility)
Construct rolling moving averages (`price_ma_3`, `price_ma_7`, `price_ma_14`, `price_ma_30`) and rolling standard deviations (`price_std_7`, `price_std_14`, `price_std_30`).

> **Leakage Prevention:**
> The rolling window spans $[t - w + 1 \dots t]$ — incorporating strictly current and past observations. No future records are included.



In [4]:
df = create_rolling_price_features(
    df, group_cols=group_cols, price_col="Modal Price",
    ma_windows=[3, 7, 14, 30],
    std_windows=[7, 14, 30]
)

roll_cols = [f"price_ma_{w}" for w in [3, 7, 14, 30]] + [f"price_std_{w}" for w in [7, 14, 30]]
display(df[df['Market'] == 'APMC Palghar'][['Price Date', 'Modal Price'] + roll_cols].iloc[30:36])



,Price Date,Modal Price,price_ma_3,price_ma_7,price_ma_14,price_ma_30,price_std_7,price_std_14,price_std_30
1180,2025-03-05,4400.0,4341.666667,4747.142857,4658.642857,4493.933333,494.586167,380.435501,523.297696
1181,2025-03-06,4500.0,4341.666667,4747.142857,4651.428571,4522.600000,494.586167,382.564366,497.842074
1182,2025-03-07,4600.0,4500.000000,4631.428571,4651.071429,4554.266667,399.215004,382.613375,469.850742
1183,2025-03-08,4400.0,4500.000000,4496.428571,4640.714286,4583.100000,249.344378,387.635124,429.995377
1184,2025-03-09,4700.0,4566.666667,4460.714286,4638.928571,4616.433333,182.492661,387.274394,396.643302
1185,2025-03-10,4300.0,4466.666667,4432.142857,4599.642857,4637.433333,190.784720,392.082528,359.770778


---
### 4. Price Momentum Features
Capture short- and medium-term price trends:
- Absolute change: $\Delta P_k = P_t - P_{t-k}$
- Percentage change: $\%\Delta P_k = \left(\frac{P_t}{P_{t-k}} - 1\right) \times 100$
for $k \in \{1, 3, 7, 14\}$.



In [5]:
df = create_price_momentum_features(df, price_col="Modal Price", windows=[1, 3, 7, 14])

mom_cols = [f"price_change_{k}" for k in [1, 3, 7, 14]] + [f"price_change_{k}_pct" for k in [1, 3, 7, 14]]
display(df[df['Market'] == 'APMC Palghar'][['Price Date', 'Modal Price'] + mom_cols].iloc[14:19])



,Price Date,Modal Price,price_change_1,price_change_3,price_change_7,price_change_14,price_change_1_pct,price_change_3_pct,price_change_7_pct,price_change_14_pct
1164,2025-02-15,4900.0,-350.0,750.0,200.0,1300.0,-6.666667,18.072289,4.255319,36.111111
1165,2025-02-16,5250.0,350.0,399.0,600.0,1610.0,7.142857,8.225108,12.903226,44.230769
1166,2025-02-17,4851.0,-399.0,-399.0,851.0,1201.0,-7.600000,-7.600000,21.275000,32.904110
1167,2025-02-18,4601.0,-250.0,-299.0,201.0,1066.0,-5.153577,-6.102041,4.568182,30.155587
1168,2025-02-19,4605.0,4.0,-645.0,455.0,905.0,0.086938,-12.285714,10.963855,24.459459


---
### 5. Price Range & Spread Features
Reflect daily market price dispersion and uncertainty:
- `price_range = Max Price - Min Price`
- `price_range_pct = (Max Price - Min Price) / Modal Price`



In [6]:
df = create_price_range_features(df, min_col="Min Price", max_col="Max Price", modal_col="Modal Price")

display(df[['Market', 'Price Date', 'Min Price', 'Modal Price', 'Max Price', 'price_range', 'price_range_pct']].head(4))



,Market,Price Date,Min Price,Modal Price,Max Price,price_range,price_range_pct
0,APMC Alibagh,2025-01-26,3000.0,3250.0,3500.0,500.0,0.153846
1,APMC Alibagh,2025-01-27,3000.0,3250.0,3500.0,500.0,0.153846
2,APMC Alibagh,2025-01-28,3000.0,3250.0,3500.0,500.0,0.153846
3,APMC Alibagh,2025-01-29,3000.0,3250.0,3500.0,500.0,0.153846


---
### 6. Calendar and Cyclical Seasonal Features
Extract discrete temporal markers (`year`, `month`, `day`, `day_of_week`, `day_of_year`, `week_of_year`, `is_weekend`) and continuous cyclical transforms:
$$\text{month\_sin} = \sin\left(\frac{2\pi \cdot \text{month}}{12}\right), \quad \text{month\_cos} = \cos\left(\frac{2\pi \cdot \text{month}}{12}\right)$$
$$\text{day\_of\_year\_sin} = \sin\left(\frac{2\pi \cdot \text{day\_of\_year}}{365.25}\right), \quad \text{day\_of\_year\_cos} = \cos\left(\frac{2\pi \cdot \text{day\_of\_year}}{365.25}\right)$$



In [7]:
df = create_calendar_features(df, date_col="Price Date")

cal_cols = ["year", "month", "day", "day_of_week", "day_of_year", "week_of_year", "is_weekend",
            "month_sin", "month_cos", "day_of_year_sin", "day_of_year_cos"]
display(df[['Price Date'] + cal_cols].head(4))



,Price Date,year,month,day,day_of_week,day_of_year,week_of_year,is_weekend,month_sin,month_cos,day_of_year_sin,day_of_year_cos
0,2025-01-26,2025,1,26,6,26,4,1,0.5,0.866025,0.432499,0.901634
1,2025-01-27,2025,1,27,0,27,5,0,0.5,0.866025,0.447945,0.894061
2,2025-01-28,2025,1,28,1,28,5,0,0.5,0.866025,0.463258,0.886224
3,2025-01-29,2025,1,29,2,29,5,0,0.5,0.866025,0.478434,0.878124


---
### 7. Market Identification Preservation
`Market`, `Variety`, and `Grade` are maintained as categorical string identifiers without arbitrary numerical encodings to facilitate categorical preprocessing during model training.



In [8]:
print("Categorical Identifiers Maintained:")
display(df[['Market', 'Variety', 'Grade']].drop_duplicates().reset_index(drop=True))



Categorical Identifiers Maintained:


,Market,Variety,Grade
0,APMC Alibagh,Other,Local
1,APMC Murud,Other,Local
2,APMC Palghar,1009 Kar,Local


---
### 8. Target Candidates Investigation
Construct forward-looking prospective prediction targets:
- `price_next_observation` ($P_{t+1}$)
- `price_after_3_observations` ($P_{t+3}$)
- `price_after_7_observations` ($P_{t+7}$)
- `future_price_change_3` ($P_{t+3} - P_t$)
- `future_price_change_7` ($P_{t+7} - P_t$)

> **CRITICAL:**
> These columns are candidate target outcomes for future supervised training and are **strictly isolated** from input features.



In [9]:
df = create_target_candidates(df, group_cols=group_cols, price_col="Modal Price")

target_cols = [
    "price_next_observation", "price_after_3_observations", "price_after_7_observations",
    "future_price_change_3", "future_price_change_7"
]

print("Target Candidates Sample (Forward Looking):")
display(df[df['Market'] == 'APMC Palghar'][['Price Date', 'Modal Price'] + target_cols].head(6))



Target Candidates Sample (Forward Looking):


,Price Date,Modal Price,price_next_observation,price_after_3_observations,price_after_7_observations,future_price_change_3,future_price_change_7
1150,2025-01-26,3600.0,3640.0,3535.0,4700.0,-65.0,1100.0
1151,2025-01-27,3640.0,3650.0,3700.0,4650.0,60.0,1010.0
1152,2025-01-28,3650.0,3535.0,3670.0,4000.0,20.0,350.0
1153,2025-01-29,3535.0,3700.0,4400.0,4400.0,865.0,865.0
1154,2025-01-30,3700.0,3670.0,4700.0,4150.0,1000.0,450.0
1155,2025-01-31,3670.0,4400.0,4650.0,4851.0,980.0,1181.0


---
### 9. Feature Leakage Audit
Exhaustive verification that no input feature utilizes future data.



In [10]:
new_features = [c for c in df.columns if c not in df_clean.columns]
input_features = [c for c in new_features if c not in target_cols]

audit_table = audit_feature_leakage(new_features, target_cols)
display(audit_table.style.set_properties(**{'text-align': 'left'}))

# Verify zero future usage in input features
leakage_violations = audit_table[(audit_table['Feature'].isin(input_features)) & (audit_table['Uses Future Info?'] == 'YES')]
print(f"Input Feature Leakage Violations Found: {len(leakage_violations)}")
assert len(leakage_violations) == 0, "Leakage detected!"
print("[PASSED] 100% of input features use strictly current and past data.")



,Feature,Information Source,Uses Future Info?,Leakage Audit Status
0,price_lag_1,Previous observations,NO,Valid Input Feature (No Leakage)
1,price_lag_2,Previous observations,NO,Valid Input Feature (No Leakage)
2,price_lag_3,Previous observations,NO,Valid Input Feature (No Leakage)
3,price_lag_7,Previous observations,NO,Valid Input Feature (No Leakage)
4,price_lag_14,Previous observations,NO,Valid Input Feature (No Leakage)
5,price_lag_30,Previous observations,NO,Valid Input Feature (No Leakage)
6,price_ma_3,Current and previous observations,NO,Valid Input Feature (No Leakage)
7,price_ma_7,Current and previous observations,NO,Valid Input Feature (No Leakage)
8,price_ma_14,Current and previous observations,NO,Valid Input Feature (No Leakage)
9,price_ma_30,Current and previous observations,NO,Valid Input Feature (No Leakage)


Input Feature Leakage Violations Found: 0
[PASSED] 100% of input features use strictly current and past data.


---
### 10. Missing Values Caused by Lags
Because we compute lags per time series group, the first $k$ observations of each group inherently have `NaN` for `price_lag_k`.
Per user instructions, we do NOT fill or interpolate these values.



In [11]:
lag_inspection = []
for k in [1, 2, 3, 7, 14, 30]:
    col = f"price_lag_{k}"
    n_na = df[col].isna().sum()
    pct_na = (n_na / len(df)) * 100
    lag_inspection.append({
        "Lag": f"lag_{k}",
        "Column": col,
        "Missing Rows": n_na,
        "Missing %": f"{pct_na:.2f}%",
        "Rows Lost if Filtered": n_na
    })

display(pd.DataFrame(lag_inspection))

print(f"Total Rows in Dataset: {len(df):,}")
print(f"Complete Cases with all 30 lags available: {len(df.dropna(subset=['price_lag_30'])):,} rows ({len(df) - df['price_lag_30'].isna().sum()} rows)")



,Lag,Column,Missing Rows,Missing %,Rows Lost if Filtered
0,lag_1,price_lag_1,3,0.18%,3
1,lag_2,price_lag_2,6,0.36%,6
2,lag_3,price_lag_3,9,0.53%,9
3,lag_7,price_lag_7,21,1.24%,21
4,lag_14,price_lag_14,42,2.49%,42
5,lag_30,price_lag_30,90,5.33%,90


Total Rows in Dataset: 1,687
Complete Cases with all 30 lags available: 1,597 rows (1597 rows)


---
### 11. Feature Summary Table
Complete metadata catalog of all constructed features and candidate targets.



In [12]:
feature_catalog = [
    # Price Lags
    {"Feature": "price_lag_1", "Type": "Lag (1 obs)", "Description": "Modal price from 1 previous observation", "Uses future information?": "NO"},
    {"Feature": "price_lag_2", "Type": "Lag (2 obs)", "Description": "Modal price from 2 previous observations", "Uses future information?": "NO"},
    {"Feature": "price_lag_3", "Type": "Lag (3 obs)", "Description": "Modal price from 3 previous observations", "Uses future information?": "NO"},
    {"Feature": "price_lag_7", "Type": "Lag (7 obs)", "Description": "Modal price from 7 previous observations", "Uses future information?": "NO"},
    {"Feature": "price_lag_14", "Type": "Lag (14 obs)", "Description": "Modal price from 14 previous observations", "Uses future information?": "NO"},
    {"Feature": "price_lag_30", "Type": "Lag (30 obs)", "Description": "Modal price from 30 previous observations", "Uses future information?": "NO"},
    # Rolling MA
    {"Feature": "price_ma_3", "Type": "Rolling Mean", "Description": "3-observation backward moving average of Modal Price", "Uses future information?": "NO"},
    {"Feature": "price_ma_7", "Type": "Rolling Mean", "Description": "7-observation backward moving average of Modal Price", "Uses future information?": "NO"},
    {"Feature": "price_ma_14", "Type": "Rolling Mean", "Description": "14-observation backward moving average of Modal Price", "Uses future information?": "NO"},
    {"Feature": "price_ma_30", "Type": "Rolling Mean", "Description": "30-observation backward moving average of Modal Price", "Uses future information?": "NO"},
    # Rolling Std
    {"Feature": "price_std_7", "Type": "Rolling Std", "Description": "7-observation rolling standard deviation (volatility)", "Uses future information?": "NO"},
    {"Feature": "price_std_14", "Type": "Rolling Std", "Description": "14-observation rolling standard deviation (volatility)", "Uses future information?": "NO"},
    {"Feature": "price_std_30", "Type": "Rolling Std", "Description": "30-observation rolling standard deviation (volatility)", "Uses future information?": "NO"},
    # Momentum Absolute & Pct
    {"Feature": "price_change_1", "Type": "Momentum", "Description": "Absolute change: Modal Price - price_lag_1", "Uses future information?": "NO"},
    {"Feature": "price_change_1_pct", "Type": "Momentum", "Description": "Percentage change over 1 previous observation", "Uses future information?": "NO"},
    {"Feature": "price_change_3", "Type": "Momentum", "Description": "Absolute change: Modal Price - price_lag_3", "Uses future information?": "NO"},
    {"Feature": "price_change_3_pct", "Type": "Momentum", "Description": "Percentage change over 3 previous observations", "Uses future information?": "NO"},
    {"Feature": "price_change_7", "Type": "Momentum", "Description": "Absolute change: Modal Price - price_lag_7", "Uses future information?": "NO"},
    {"Feature": "price_change_7_pct", "Type": "Momentum", "Description": "Percentage change over 7 previous observations", "Uses future information?": "NO"},
    {"Feature": "price_change_14", "Type": "Momentum", "Description": "Absolute change: Modal Price - price_lag_14", "Uses future information?": "NO"},
    {"Feature": "price_change_14_pct", "Type": "Momentum", "Description": "Percentage change over 14 previous observations", "Uses future information?": "NO"},
    # Price Range
    {"Feature": "price_range", "Type": "Price Range", "Description": "Intraday price spread: Max Price - Min Price", "Uses future information?": "NO"},
    {"Feature": "price_range_pct", "Type": "Price Range", "Description": "Relative intraday spread: (Max - Min) / Modal Price", "Uses future information?": "NO"},
    # Calendar & Cyclical
    {"Feature": "year", "Type": "Calendar", "Description": "Observation calendar year (2025–2026)", "Uses future information?": "NO"},
    {"Feature": "month", "Type": "Calendar", "Description": "Observation calendar month (1–12)", "Uses future information?": "NO"},
    {"Feature": "day", "Type": "Calendar", "Description": "Observation day of month (1–31)", "Uses future information?": "NO"},
    {"Feature": "day_of_week", "Type": "Calendar", "Description": "Day of week index (0=Monday, 6=Sunday)", "Uses future information?": "NO"},
    {"Feature": "day_of_year", "Type": "Calendar", "Description": "Day of year (1–366)", "Uses future information?": "NO"},
    {"Feature": "week_of_year", "Type": "Calendar", "Description": "ISO week of year (1–53)", "Uses future information?": "NO"},
    {"Feature": "is_weekend", "Type": "Calendar Indicator", "Description": "Binary flag (1 if Saturday or Sunday, 0 otherwise)", "Uses future information?": "NO"},
    {"Feature": "month_sin", "Type": "Cyclical Seasonality", "Description": "Sine transform of month: sin(2*pi*month/12)", "Uses future information?": "NO"},
    {"Feature": "month_cos", "Type": "Cyclical Seasonality", "Description": "Cosine transform of month: cos(2*pi*month/12)", "Uses future information?": "NO"},
    {"Feature": "day_of_year_sin", "Type": "Cyclical Seasonality", "Description": "Sine transform of day of year", "Uses future information?": "NO"},
    {"Feature": "day_of_year_cos", "Type": "Cyclical Seasonality", "Description": "Cosine transform of day of year", "Uses future information?": "NO"},
    # Target Candidates
    {"Feature": "price_next_observation", "Type": "Target Candidate", "Description": "Lead 1: Modal Price of next observation (t+1)", "Uses future information?": "YES (TARGET)"},
    {"Feature": "price_after_3_observations", "Type": "Target Candidate", "Description": "Lead 3: Modal Price after 3 observations (t+3)", "Uses future information?": "YES (TARGET)"},
    {"Feature": "price_after_7_observations", "Type": "Target Candidate", "Description": "Lead 7: Modal Price after 7 observations (t+7)", "Uses future information?": "YES (TARGET)"},
    {"Feature": "future_price_change_3", "Type": "Target Candidate", "Description": "Price difference: P(t+3) - P(t)", "Uses future information?": "YES (TARGET)"},
    {"Feature": "future_price_change_7", "Type": "Target Candidate", "Description": "Price difference: P(t+7) - P(t)", "Uses future information?": "YES (TARGET)"},
]

catalog_df = pd.DataFrame(feature_catalog)
display(catalog_df.style.set_properties(**{'text-align': 'left'}))



,Feature,Type,Description,Uses future information?
0,price_lag_1,Lag (1 obs),Modal price from 1 previous observation,NO
1,price_lag_2,Lag (2 obs),Modal price from 2 previous observations,NO
2,price_lag_3,Lag (3 obs),Modal price from 3 previous observations,NO
3,price_lag_7,Lag (7 obs),Modal price from 7 previous observations,NO
4,price_lag_14,Lag (14 obs),Modal price from 14 previous observations,NO
5,price_lag_30,Lag (30 obs),Modal price from 30 previous observations,NO
6,price_ma_3,Rolling Mean,3-observation backward moving average of Modal Price,NO
7,price_ma_7,Rolling Mean,7-observation backward moving average of Modal Price,NO
8,price_ma_14,Rolling Mean,14-observation backward moving average of Modal Price,NO
9,price_ma_30,Rolling Mean,30-observation backward moving average of Modal Price,NO


---
### 12. Save Engineered Dataset



In [13]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Successfully saved engineered dataset ({len(df):,} rows x {len(df.columns)} cols) to:")
print(f"  {OUTPUT_PATH}")



Successfully saved engineered dataset (1,687 rows x 52 cols) to:
  /Users/moksh/Desktop/MandiMitra-ML/data/processed/maharashtra_rice_features.csv


---
### 13. Step 5 Final Engineering Summary Report



In [14]:
print("=" * 75)
print("STEP 5 FEATURE ENGINEERING REPORT")
print("=" * 75)
print(f"Rows before feature engineering       : {len(df_clean):,}")
print(f"Rows after feature engineering        : {len(df):,}")
print(f"Total features created (input features): {len(input_features)}")
print(f"Target candidates created             : {len(target_cols)}")
print(f"Rows lost when requiring lag_30       : {df['price_lag_30'].isna().sum()} (30 per retained group)")
print(f"Data leakage detected                 : ZERO (All inputs use strictly past/current info)")
print(f"Missing values created in lag_1       : {df['price_lag_1'].isna().sum()} (1 per group)")
print(f"Missing values created in lag_7       : {df['price_lag_7'].isna().sum()} (7 per group)")
print(f"Missing values created in lag_14      : {df['price_lag_14'].isna().sum()} (14 per group)")
print(f"Missing values created in lag_30      : {df['price_lag_30'].isna().sum()} (30 per group)")
print("=" * 75)
print("\nEngineered Input Features List:")
for idx, f in enumerate(input_features, 1):
    print(f"  {idx:>2}. {f}")
print("\nTarget Candidates List:")
for idx, t in enumerate(target_cols, 1):
    print(f"  {idx:>2}. {t}")



STEP 5 FEATURE ENGINEERING REPORT
Rows before feature engineering       : 1,687
Rows after feature engineering        : 1,687
Total features created (input features): 34
Target candidates created             : 5
Rows lost when requiring lag_30       : 90 (30 per retained group)
Data leakage detected                 : ZERO (All inputs use strictly past/current info)
Missing values created in lag_1       : 3 (1 per group)
Missing values created in lag_7       : 21 (7 per group)
Missing values created in lag_14      : 42 (14 per group)
Missing values created in lag_30      : 90 (30 per group)

Engineered Input Features List:
   1. price_lag_1
   2. price_lag_2
   3. price_lag_3
   4. price_lag_7
   5. price_lag_14
   6. price_lag_30
   7. price_ma_3
   8. price_ma_7
   9. price_ma_14
  10. price_ma_30
  11. price_std_7
  12. price_std_14
  13. price_std_30
  14. price_change_1
  15. price_change_1_pct
  16. price_change_3
  17. price_change_3_pct
  18. price_change_7
  19. price_change_7_